# KoELECTRA 분류 모델 학습 (Google Colab)

lotte-insight 프로젝트 — `training/train_classifier.py` Colab 실행용 노트북

**모델:** `monologg/koelectra-small-v3-discriminator` fine-tuning  
**분류 방식:** 멀티라벨 (BCEWithLogitsLoss + per-label pos_weight)  
**라벨:** INJURY_ROSTER / TRANSACTION_CONTRACT / MATCH_RELATED / PERFORMANCE_ANALYSIS / INTERVIEW / CLUB_OPERATION / ETC  
**목표:** val macro-F1 ≥ 0.70

**사전 준비 (Google Drive 업로드)**
```
MyDrive/lotte-insight-data/
  labeled_titles.csv
  labeled_players.csv
```

In [ ]:
# 1. GPU 확인
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA:', torch.version.cuda)
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('[WARN] GPU not available — training will be very slow on CPU')

In [ ]:
# 2. 학습 데이터 업로드 (로컬 → Colab 직접 전송)
# 실행 후 파일 선택 창에서 아래 두 파일을 선택하시오:
#   - labeled_titles.csv
#   - labeled_players.csv
import os
from google.colab import files

DATA_DIR = '/content/lotte-insight/training/data'
os.makedirs(DATA_DIR, exist_ok=True)

uploaded = files.upload()  # 파일 선택 창 열림

import shutil
for fname, content in uploaded.items():
    dst = f'{DATA_DIR}/{os.path.basename(fname)}'
    with open(dst, 'wb') as f:
        f.write(content)
    print(f'저장 완료: {dst}  ({len(content):,} bytes)')

In [ ]:
# 3. 레포 클론 및 의존성 설치
# GitHub URL을 본인 레포로 교체하시오
GITHUB_REPO_URL = 'https://github.com/YOUR_USERNAME/lotte-insight.git'

!git clone {GITHUB_REPO_URL} /content/lotte-insight
%cd /content/lotte-insight/training
!pip install -q -r requirements.txt

In [ ]:
# 5. 학습 데이터 분포 확인
import os
import pandas as pd
from collections import Counter

DATA_DIR = '/content/lotte-insight/training/data'
LABELS = ['MATCH_RELATED','INJURY_ROSTER','TRANSACTION_CONTRACT',
          'PERFORMANCE_ANALYSIS','INTERVIEW','CLUB_OPERATION','ETC']

total_dist = Counter()
grand_total = 0

for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        print(f'[MISSING] {fname}')
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
    dist = Counter(lotte['primary_label'].dropna())
    total_dist.update(dist)
    grand_total += len(lotte)
    print(f'{fname}: {len(lotte)}행')

print(f'\n합산 학습가능 행: {grand_total}')
print('\n라벨 분포:')
for label in LABELS:
    count = total_dist.get(label, 0)
    bar = '█' * (count // 50)
    print(f'  {label:<25} {count:>5}  {bar}')

minority = [l for l in LABELS if total_dist.get(l, 0) < 100]
if minority:
    print(f'\n[주의] 100건 미만 라벨: {minority} — pos_weight로 보정되나 F1 불안정 가능')
# event_summary 커버리지 확인 (학습/추론 불일치 주의)
print()
all_lotte = []
for fname in ['labeled_titles.csv', 'labeled_players.csv']:
    path = f'{DATA_DIR}/{fname}'
    if not os.path.exists(path):
        continue
    df = pd.read_csv(path, encoding='utf-8-sig')
    lotte = df[df['is_lotte_related'].astype(str).str.lower() == 'true']
    all_lotte.append(lotte)

if all_lotte:
    combined = pd.concat(all_lotte, ignore_index=True)
    has_summary = combined['event_summary'].fillna('').astype(str).str.strip().ne('') if 'event_summary' in combined.columns else pd.Series([False]*len(combined))
    count = int(has_summary.sum())
    pct = count / len(combined) * 100 if len(combined) else 0
    print(f'event_summary 채워진 행: {count}/{len(combined)} ({pct:.0f}%)')
    if count > 0:
        print('[주의] event_summary는 분류 이전에 존재하지 않는 값입니다.')
        print('       학습 시 auxiliary_text에 포함되면 학습/추론 불일치(train-inference mismatch)가 발생합니다.')
        print('       train_classifier.py의 build_auxiliary_text()는 description_snippet만 사용하도록 유지하세요.')


In [ ]:
# 6. 학습 실행
# best model: val macro-F1 기준 자동 저장
# T4 GPU 기준 약 5~10분 소요
!python train_classifier.py \
    --epochs 5 \
    --lr 5e-5 \
    --batch 16

In [ ]:
# 7. 학습 결과 확인 — per-label 임계값 및 전체 평가
import json, os

threshold_path = '/content/lotte-insight/training/models/classifier_koelectra/label_thresholds.json'
if os.path.exists(threshold_path):
    with open(threshold_path) as f:
        thresholds = json.load(f)
    print('Per-label 최적 임계값:')
    for label, t in thresholds.items():
        print(f'  {label:<25} {t:.2f}')
else:
    print('[WARN] label_thresholds.json 없음')

print()
!python train_classifier.py --eval-only

In [ ]:
# 8. 학습된 모델 다운로드 (Colab → 로컬)
import shutil, os
from google.colab import files

LOCAL_MODEL_DIR = '/content/lotte-insight/training/models/classifier_koelectra'
ZIP_PATH = '/content/classifier_koelectra.zip'

if os.path.exists(LOCAL_MODEL_DIR):
    shutil.make_archive('/content/classifier_koelectra', 'zip', LOCAL_MODEL_DIR)
    print(f'압축 완료: {ZIP_PATH}')
    files.download(ZIP_PATH)  # 로컬로 다운로드
else:
    print('[ERROR] 모델 디렉토리 없음 — 학습 실패 여부 확인 필요')

In [ ]:
# 9. 빠른 추론 테스트
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/lotte-insight/training/models/classifier_koelectra'
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)
model.eval()

with open(f'{MODEL_DIR}/label_encoder.json', encoding='utf-8') as f:
    LABELS = json.load(f)
with open(f'{MODEL_DIR}/label_thresholds.json', encoding='utf-8') as f:
    THRESHOLDS = json.load(f)

def predict(title: str, description: str = '', event_summary: str = '') -> list[str]:
    auxiliary = description
    if event_summary:
        auxiliary = f'{description} [요약정보] 요약: {event_summary}' if description else f'요약: {event_summary}'
    enc = tokenizer(title, auxiliary, truncation='only_second', padding='max_length',
                    max_length=256, return_tensors='pt')
    with torch.no_grad():
        logits = model(**enc).logits[0]
    probs = torch.sigmoid(logits)
    result = [LABELS[i] for i, p in enumerate(probs) if p.item() >= THRESHOLDS.get(LABELS[i], 0.5)]
    return result if result else ['ETC']

SAMPLES = [
    ('롯데 나균안, 시즌 5승 달성…선발 로테이션 안정화', '나균안이 두산전 6이닝 2실점 호투로 시즌 5승을 따냈다.'),
    ('롯데 전준우 햄스트링 부상…2주 결장 예상', '전준우가 경기 중 부상으로 1군 엔트리에서 말소됐다.'),
    ('롯데 구단, 외국인 투수 교체 결정', '롯데가 부진한 외국인 투수를 방출하고 새 용병을 물색 중이다.'),
    ('서튼 감독 "선수들이 잘 따라줬다"', '래리 서튼 감독이 경기 후 인터뷰에서 선수단을 칭찬했다.'),
    ('사직구장 개막 이벤트, 팬 3만 명 몰려', '롯데 자이언츠가 홈 개막전 기념 이벤트를 성황리에 개최했다.'),
]

print('추론 결과:')
for title, desc in SAMPLES:
    predicted = predict(title, desc)
    print(f'  [{", ".join(predicted)}] {title}')

# 10. (선택) macro-F1 미달 시 에폭 추가 학습
# macro-F1 < 0.70 이면 아래 실행
# !python train_classifier.py --epochs 8 --lr 3e-5 --batch 16